# Delivered Image Quality Trending (v1)

**Author:** Keith Bechtol, Aaron Roodman  
**Date Created:** 2026-05-04  
**Last Modified:** 2026-05-04  
**Status:** In Progress  
**Keywords:** image quality, PSF FWHM, ellipticity, AOS, donut blur, moment score, ConsDB, trending  

## Description

Trending of delivered image quality metrics for LSSTCam survey visits, queried from the ConsDB.

Key functionality:
1. Pull per-detector PSF moments and per-visit AOS / atmosphere metrics from ConsDB
2. Aggregate to per-visit and per-night summaries
3. Plot distributions and time-series of FWHM, ellipticity, and the moment score

**Output:** Both the parquet cache `ccdvisits_{day_obs_min}-{day_obs_max}.pq` and the PDF figures are written to the path in `output_dir` (default `blocks/output/`).

**Based on:** `image_quality_trending.ipynb` by Keith Bechtol. The single-shot 4-way ConsDB join was replaced with a chunked, explicit-join, dtype-aware fetch in `code/consdb_fetch.py` so multi-week date ranges load without OOM.

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-04 | Keith Bechtol | Original notebook |
| 2026-05-04 | Aaron Roodman | Adopted rubin-work template; swapped `fetch()` for chunked, explicit-join version in `code/consdb_fetch.py` to fix OOM on multi-week date ranges; removed `fetchA` / `fetchB` workaround |
| 2026-05-04 | Aaron Roodman | Cleanup: route parquet cache and PDF figures through `output_dir` (default `blocks/output/`); safer `no_proxy` mutation; tqdm progress bar in `fetch_chunked`; removed unused vars and dead commented-out plot blocks |
| 2026-05-04 | Aaron Roodman | Add per-chunk parquet caching in `fetch_chunked` (cache_dir kwarg); switch 95-5 FWHM and ellipticity to simple differences (was quadrature); add `moment_score` to `visits_summary`; add 7-quantity per-visit corner plot |
| 2026-05-04 | Aaron Roodman | Corner plot: 2D log-density (`hist2d` + `LogNorm`) instead of scatter; quantity titles atop diagonal histograms; `log10(moment_score)`; second page with `donut_blur_fwhm < 0.8` cut, both pages in one PDF |
| 2026-05-04 | Aaron Roodman | Add FBS stability BLOCK-T614/T698/T703/T706 to `SURVEY_PROGRAMS`; introduce `FWHM_95_05_QUADRATURE` parameter (default `True`); add combined per-visit IQ score (`aos_cam_fwhm` + `psf_fwhm_95_05` + `psf_e_50` + `psf_e_95_05` percentile-averaged, 99 = best), with per-band histograms and 10/50/90 nightly time-series |
| 2026-05-04 | Aaron Roodman | Add 10/50/90 percentile-per-night time-series for each of the four IQ score input components (`aos_cam_fwhm`, `psf_fwhm_95_05`, `psf_e_50`, `psf_e_95_05`) |
| 2026-05-04 | Aaron Roodman | Add per-night violin-plot time-series (10/50/90 percentile lines) alongside the existing errorbar versions for the combined IQ score and each of the four IQ component variables; new `plot_violin_timeseries` helper |
| 2026-05-04 | Aaron Roodman | Cache-first data load: notebook calls new `try_load_from_cache(...)` before touching ConsDB; if every 7-day chunk is on disk the run skips the `ConsDbClient` construction entirely |
| 2026-05-04 | Aaron Roodman | Merge TQ's shapelets score (`shapelets_score_all_visits.pq`) into `visits_summary` as a per-visit median over the same detector list, gated on `USE_SHAPELETS_SCORE` (default `True`); include `shapelets_score` as a 5th IQ score component when present, with matching per-component time-series and violin plots |
| 2026-05-04 | Aaron Roodman | Memory refactor: build `visits_summary` chunk-by-chunk via new `fetch_visits_summary_chunked` / `try_load_visits_summary_from_cache`; drop the gigabyte post-derivation `ccdvisits` parquet save/load that was OOM'ing; load a slim `ccdvisits` (just the per-detector plot columns) via `chunkwise_columns` when `LOAD_CCDVISITS_LITE` is True. Also add normalized inverse CDF plot for each IQ score component |
| 2026-05-04 | Aaron Roodman | Reorganize plots: Keith's originals consolidated into a single multi-page PDF; IQ score pipeline plots consolidated into a single multi-page PDF, reordered as validation/CDF -> per-component time-series -> combined IQ |


## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Data Access](#data)
5. [Analysis](#analysis)
6. [Results & Plots](#results)


<a id='params'></a>
## Parameters

In [ ]:
# Parameters cell. Set defaults here

#day_obs_min = 20250620 # Start of SV surveys
#day_obs_max = 20250921 # Last night of SV surveys
#day_obs_min = 20251026 # Start of Early Operations
day_obs_min = 20260301 # March
#day_obs_min = 20260414 # Start of selection period
day_obs_max = 20260423 # End of selection period
#day_obs_max = 20260425

In [ ]:
from pathlib import Path

SAVE_FIGURES = True
SURVEY_PROGRAMS = [
    'BLOCK-365', 'BLOCK-407', 'BLOCK-408', 'BLOCK-416', 'BLOCK-417',
    'BLOCK-419', 'BLOCK-421',                                 # Commissioning
    'BLOCK-T614', 'BLOCK-T698', 'BLOCK-T703', 'BLOCK-T706',   # FBS stability
]
#SURVEY_PROGRAMS = ['BLOCK-407', 'BLOCK-408', 'BLOCK-416', 'BLOCK-417', 'BLOCK-419', 'BLOCK-421']  # Early Ops
CAM_FWHM = 0.207  # arcsec

# 95-5 PSF FWHM variation per visit:
#   True  -> sqrt(fwhm_95**2 - fwhm_05**2)  (quadrature, default)
#   False -> fwhm_95 - fwhm_05              (simple difference)
# Ellipticity 95-5 is always a simple difference (quadrature ill-defined).
FWHM_95_05_QUADRATURE = True

# Merge TQ's shapelets score (per-visit median over the same
# detector list used in the ConsDB query) into visits_summary and
# include it as a 5th IQ score component. Set False to skip.
USE_SHAPELETS_SCORE = True
SHAPELETS_TABLE_PATH = '/sdf/data/rubin/user/ztq1996/psf-rubin/psf_zernike/data/shapelets_score_all_visits.pq'

# The full per-CCD `ccdvisits` table is gigabytes; we don't build
# it by default. The per-detector plots load just the columns they
# need via `chunkwise_columns` when this flag is True. Set False
# to skip those plots entirely.
LOAD_CCDVISITS_LITE = True
CCDVISITS_LITE_COLUMNS = (
    'visit_id', 'detector', 'day_obs', 'band', 'seq_num',
    'donut_blur_fwhm',
    'psf_fwhm', 'psf_e', 'moment_score',  # derived in chunkwise_columns
)

# Parquet cache lives in <topic>/output/ per rubin-work convention.
output_dir = Path('output')
output_dir.mkdir(parents=True, exist_ok=True)

# Figures: prefer output/, fall back to figures/, then cwd.
for _candidate in (Path('output'), Path('figures'), Path('.')):
    if _candidate.is_dir():
        figures_dir = _candidate
        break


<a id='setup'></a>
## Setup & Imports

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

#%matplotlib widget

In [ ]:
from lsst.utils.plotting import publication_plots
from lsst.utils.plotting import get_multiband_plot_colors

colors = get_multiband_plot_colors()
bands = colors.keys()

In [ ]:
publication_plots.set_rubin_plotstyle()

In [ ]:
from lsst.summit.utils import (
    getAirmassSeeingCorrection,
    getBandpassSeeingCorrection,
)

In [ ]:
import os
os.environ["no_proxy"] = os.environ.get("no_proxy", "") + ",.consdb"

# ConsDbClient is constructed lazily in the Data Access section, only
# if the chunk cache is incomplete. That way a fully cached run never
# touches the network and never even imports lsst.summit.utils.ConsDbClient.
consdb_url = "http://consdb-pq.consdb:8080/consdb"


In [ ]:
instrument = 'lsstcam'

<a id='functions'></a>
## Helper Functions

In [ ]:
import sys
from pathlib import Path

# Make code/ importable from this notebook
sys.path.insert(0, str(Path.cwd() / 'code'))
from consdb_fetch import (
    fetch_visits_summary_chunked,
    try_load_visits_summary_from_cache,
    chunkwise_columns,
)

In [ ]:
from astropy.time import Time


def dayObsToTime(day_obs):
    """Convert a YYYYMMDD day_obs integer to an astropy Time at noon UTC."""
    s = str(int(day_obs))
    return Time(f'{s[0:4]}-{s[4:6]}-{s[6:8]} 12:00:00', format='iso')


In [ ]:
from matplotlib.colors import LogNorm


def corner_plot(df, columns, labels=None, bins=40,
                cmap='viridis', density_log=True,
                quantile_clip=(0.5, 99.5), figsize_per_var=2.0):
    """Lower-triangle corner plot: 2D density (log) + diagonal histograms.

    Each lower-triangle panel is a `hist2d` density map (counts colored
    by `cmap`, optionally on a log scale). Each diagonal panel is a 1D
    histogram of the column, with the variable label printed as the
    panel title so it sits *above* the histogram (and therefore above
    every column it heads). Empty 2D bins are transparent so the
    background stays clean.

    Returns (fig, axes).
    """
    n = len(columns)
    labels = list(labels) if labels is not None else list(columns)

    series = [pd.to_numeric(df[c], errors='coerce') for c in columns]
    ranges = []
    for s in series:
        clean = s.dropna()
        if len(clean):
            lo, hi = np.nanpercentile(clean, quantile_clip)
        else:
            lo, hi = 0.0, 1.0
        ranges.append((lo, hi))

    fig, axes = plt.subplots(
        n, n,
        figsize=(figsize_per_var * n, figsize_per_var * n),
        squeeze=False,
    )
    norm = LogNorm() if density_log else None

    for i in range(n):
        for j in range(n):
            ax = axes[i, j]
            if j > i:
                ax.set_visible(False)
                continue
            xj = series[j]
            if i == j:
                clean = xj.dropna()
                ax.hist(clean, bins=bins, range=ranges[j],
                        histtype='step', color='black')
                ax.set_yticks([])
                ax.set_title(labels[j], fontsize=9)
            else:
                yi = series[i]
                m = xj.notna() & yi.notna()
                ax.hist2d(
                    xj[m].to_numpy(), yi[m].to_numpy(),
                    bins=bins, range=[ranges[j], ranges[i]],
                    cmap=cmap, norm=norm, cmin=1,
                )
                ax.set_ylim(ranges[i])
            ax.set_xlim(ranges[j])
            if j == 0 and i > 0:
                ax.set_ylabel(labels[i], fontsize=9)
            else:
                ax.tick_params(labelleft=False)
            if i == n - 1:
                ax.set_xlabel(labels[j], fontsize=9)
                ax.tick_params(axis='x', rotation=45)
            else:
                ax.tick_params(labelbottom=False)

    return fig, axes


In [ ]:
def plot_violin_timeseries(df, col, label, sel=None, color='tab:blue',
                           figsize=(15, 6), dpi=200, ylim_low=0,
                           min_n_per_night=2):
    """Per-night violin distributions of `df[col]` with 10/50/90 lines.

    Each violin is a KDE of the per-visit values for that night.
    Black horizontal ticks across each body mark the 10th, 50th, and
    90th percentiles. Nights with fewer than `min_n_per_night` visits
    are skipped (a KDE needs >= 2 points).
    """
    if sel is None:
        sel = pd.Series(True, index=df.index)
    keep = sel & df[col].notna() & df['day_obs'].notna()
    sub = df.loc[keep, [col, 'day_obs']]

    days = sorted(sub['day_obs'].unique())
    data_per_day, positions, day_labels = [], [], []
    for d in days:
        arr = sub.loc[sub['day_obs'] == d, col].to_numpy()
        if len(arr) >= min_n_per_night:
            data_per_day.append(arr)
            positions.append(dayObsToTime(d).mjd)
            day_labels.append(int(d))

    if not data_per_day:
        print(f"plot_violin_timeseries: no nights with >= "
              f"{min_n_per_night} visits for {col}")
        return None

    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
    parts = ax.violinplot(
        data_per_day, positions=positions, widths=0.6,
        showmedians=False, showextrema=False,
        quantiles=[[0.10, 0.50, 0.90]] * len(data_per_day),
    )
    for body in parts['bodies']:
        body.set_facecolor(color)
        body.set_edgecolor(color)
        body.set_alpha(0.6)
    if 'cquantiles' in parts:
        parts['cquantiles'].set_color('black')
        parts['cquantiles'].set_linewidth(0.8)

    ax.set_xticks(positions)
    ax.set_xticklabels(day_labels, rotation=90, fontsize=7.5)
    ax.set_xlim(positions[0] - 1, positions[-1] + 1)
    if ylim_low is not None:
        ax.set_ylim(ylim_low, ax.get_ylim()[1])
    ax.set_ylabel(label)
    ax.set_title(f'Per-Night Distributions (violin, 10/50/90): {label}')
    ax.grid(axis='y')
    ax.minorticks_off()
    fig.tight_layout()
    return fig


<a id='data'></a>
## Data Access

In [ ]:
# Cache-first per-visit summary load.
#
# fetch_visits_summary_chunked iterates the chunks (using the same
# parquet cache key as the per-CCD fetcher), runs groupby('visit_id')
# on each chunk, and discards the per-CCD frame between chunks. The
# concatenated result is the per-visit `visits_summary` -- much smaller
# than the full per-CCD `ccdvisits`. CAM_FWHM- and airmass-dependent
# derivations happen on this small frame in the Analysis section below.
visits_summary = try_load_visits_summary_from_cache(
    output_dir, instrument, SURVEY_PROGRAMS,
    day_obs_min, day_obs_max, chunk_days=7,
)

if visits_summary is None:
    print('Chunk cache incomplete; connecting to ConsDB...')
    from lsst.summit.utils import ConsDbClient
    client = ConsDbClient(consdb_url)
    visits_summary = fetch_visits_summary_chunked(
        client, instrument, day_obs_min, day_obs_max,
        SURVEY_PROGRAMS, chunk_days=7, cache_dir=output_dir,
    )
else:
    print(f'Loaded {len(visits_summary):,} visits from chunk cache '
          '(no ConsDB needed)')

print(f'visits_summary: {len(visits_summary):,} visits')


### Per-CCD columns (optional, for per-detector plots)

In [ ]:
# Lazy, narrow per-CCD load for the per-detector histogram plots below.
# Pulls only the columns those plots need (`CCDVISITS_LITE_COLUMNS`),
# reading each chunk parquet one at a time so peak memory stays bounded.
# Set `LOAD_CCDVISITS_LITE = False` in the Parameters cell to skip both
# this load and the per-detector plots downstream.
if LOAD_CCDVISITS_LITE:
    ccdvisits = chunkwise_columns(
        output_dir, instrument, SURVEY_PROGRAMS,
        day_obs_min, day_obs_max,
        columns=list(CCDVISITS_LITE_COLUMNS),
        chunk_days=7,
    )
    print(f'ccdvisits (lite): {len(ccdvisits):,} rows, '
          f'{ccdvisits.memory_usage(deep=True).sum()/1e6:.0f} MB')
else:
    ccdvisits = None
    print('LOAD_CCDVISITS_LITE = False -- skipping per-detector plots')


<a id='analysis'></a>
## Analysis

### Per-visit summary table

In [ ]:
# Per-visit derivations on the already-built `visits_summary`.
#
# `visits_summary` arrives with the per-CCD aggregations done
# (median, 0.05/0.50/0.95 quantiles) plus the per-visit columns
# inherited via .first(). Here we add the small post-processing layer
# that depends on notebook-level constants (CAM_FWHM,
# FWHM_95_05_QUADRATURE) or external corrections.
airmass_corr = np.array(
    [getAirmassSeeingCorrection(a) for a in visits_summary['airmass']]
)
bandpass_corr = np.array(
    [getBandpassSeeingCorrection(b) for b in visits_summary['physical_filter']]
)
visits_summary['psf_fwhm_zenith_500nm_50'] = (
    visits_summary['psf_fwhm_50'] * airmass_corr * bandpass_corr
)

visits_summary['donut_blur_atm_fwhm'] = np.sqrt(
    visits_summary['donut_blur_fwhm']**2 - CAM_FWHM**2
)
visits_summary['aos_cam_fwhm'] = np.sqrt(
    visits_summary['aos_fwhm']**2 + CAM_FWHM**2
)

# 95% - 5% spread across CCDs within each visit.
# FWHM: quadrature or simple difference per FWHM_95_05_QUADRATURE.
# Ellipticity: simple difference always (quadrature ill-defined).
if FWHM_95_05_QUADRATURE:
    visits_summary['psf_fwhm_95_05'] = np.sqrt(
        visits_summary['psf_fwhm_95']**2 - visits_summary['psf_fwhm_05']**2
    )
else:
    visits_summary['psf_fwhm_95_05'] = (
        visits_summary['psf_fwhm_95'] - visits_summary['psf_fwhm_05']
    )
visits_summary['psf_e_95_05'] = (
    visits_summary['psf_e_95'] - visits_summary['psf_e_05']
)

visits_summary['sys_50'] = np.sqrt(
    visits_summary['psf_fwhm_50']**2 + CAM_FWHM**2
    - visits_summary['donut_blur_fwhm']**2
)
visits_summary['psf_fwhm_model'] = np.sqrt(
    visits_summary['aos_fwhm']**2 + visits_summary['donut_blur_fwhm']**2
)


### Shapelets score (TQ)

In [ ]:
# Merge in TQ's per-detector shapelets score, aggregated to a per-visit
# median over the same detector list used in the ConsDB query
# (vignetted detectors and the corner wavefront sensors are dropped).
# visit_id = day_obs * 100000 + seq_num, matching the ConsDB convention.
visits_summary['shapelets_score'] = np.nan

if USE_SHAPELETS_SCORE:
    try:
        from consdb_fetch import VIGNETTED_DETECTORS, CORNER_WAVEFRONT_DETECTORS
        excluded = set(VIGNETTED_DETECTORS) | set(CORNER_WAVEFRONT_DETECTORS)

        shapelets = pd.read_parquet(SHAPELETS_TABLE_PATH)
        shapelets = shapelets[~shapelets['detector_id'].isin(excluded)]
        per_visit = (
            shapelets.groupby('visit_id')['shapelets_score'].median()
        )
        visits_summary['shapelets_score'] = (
            visits_summary.index.map(per_visit)
        )

        n_with = visits_summary['shapelets_score'].notna().sum()
        print(
            f"Shapelets score merged: {n_with} / {len(visits_summary)} "
            f"visits have a per-visit median score "
            f"(table covers visits up to ~2026-04-15)"
        )
    except FileNotFoundError as exc:
        print(f"Shapelets table not found, skipping: {exc}")


### Per-night summary table

In [ ]:
selection = (visits_summary['band'] != "y") & ~visits_summary['observation_reason'].str.contains('filter_change_close_loop')
selection = selection & ~((visits_summary['day_obs'] == 20260424) & (visits_summary['seq_num'] < 398)) # Remove corrupted period due to lack of hexapod compensation

groups = visits_summary[selection].groupby('day_obs')
day_obs_summary = pd.DataFrame({
    'day_obs': groups['day_obs'].first(),
    'n_visits': groups['day_obs'].count(),
    'psf_fwhm_95_05_low': groups['psf_fwhm_95_05'].quantile(0.10),
    'psf_fwhm_95_05_50': groups['psf_fwhm_95_05'].quantile(0.50),
    'psf_fwhm_95_05_high': groups['psf_fwhm_95_05'].quantile(0.90),
    'aos_fwhm_low': groups['aos_fwhm'].quantile(0.10),
    'aos_fwhm_50': groups['aos_fwhm'].quantile(0.50),
    'aos_fwhm_high': groups['aos_fwhm'].quantile(0.90),
    'aos_cam_fwhm_low': groups['aos_cam_fwhm'].quantile(0.10),
    'aos_cam_fwhm_50': groups['aos_cam_fwhm'].quantile(0.50),
    'aos_cam_fwhm_high': groups['aos_cam_fwhm'].quantile(0.90),
    'sys_50_low': groups['sys_50'].quantile(0.10),
    'sys_50_50': groups['sys_50'].quantile(0.50),
    'sys_50_high': groups['sys_50'].quantile(0.90),
    'psf_e_50_low': groups['psf_e_50'].quantile(0.10),
    'psf_e_50_50': groups['psf_e_50'].quantile(0.50),
    'psf_e_50_high': groups['psf_e_50'].quantile(0.90),
    'psf_e1_50_low': groups['psf_e1_50'].quantile(0.10),
    'psf_e1_50_50': groups['psf_e1_50'].quantile(0.50),
    'psf_e1_50_high': groups['psf_e1_50'].quantile(0.90),
    'psf_e2_50_low': groups['psf_e2_50'].quantile(0.10),
    'psf_e2_50_50': groups['psf_e2_50'].quantile(0.50),
    'psf_e2_50_high': groups['psf_e2_50'].quantile(0.90),
    'psf_fwhm_50_low': groups['psf_fwhm_50'].quantile(0.10),
    'psf_fwhm_50_50': groups['psf_fwhm_50'].quantile(0.50),
    'psf_fwhm_50_high': groups['psf_fwhm_50'].quantile(0.90),
    'donut_blur_atm_fwhm_low': groups['donut_blur_atm_fwhm'].quantile(0.10),
    'donut_blur_atm_fwhm_50': groups['donut_blur_atm_fwhm'].quantile(0.50),
    'donut_blur_atm_fwhm_high': groups['donut_blur_atm_fwhm'].quantile(0.90),
})

In [ ]:
# Combined per-visit Image Quality score.
#
# For each of the four IQ_COMPONENTS, rank visits from best (lowest
# value) to worst, scaled to 0-99 (99 = best). Average the four
# percentiles per visit to yield a combined score on the same scale.
#
# Percentiles are computed over the same `iq_selection` quality cut
# used downstream (non-y, not filter-change, not the 2026-04-24 corrupt
# block) so the ensemble is the trended science sample.
IQ_COMPONENTS = [
    'aos_cam_fwhm',
    'psf_fwhm_95_05',
    'psf_e_50',
    'psf_e_95_05',
]
# moment_score is excluded -- coma/trefoil aren't populated in ConsDB
# before 2026-03, which would drop those visits from the score.
# Shapelets score, on the other hand, is added when available even
# though TQ's table currently only runs through ~2026-04-15;
# `mean(skipna=True)` per visit gracefully averages over whichever
# components have a value.
if USE_SHAPELETS_SCORE and visits_summary['shapelets_score'].notna().any():
    IQ_COMPONENTS.append('shapelets_score')
print('IQ components:', IQ_COMPONENTS)

iq_selection = (visits_summary['band'] != 'y') \
    & ~visits_summary['observation_reason'].str.contains('filter_change_close_loop')
iq_selection = iq_selection & ~(
    (visits_summary['day_obs'] == 20260424) & (visits_summary['seq_num'] < 398)
)


def _ranking_percentile(s):
    """Rank to 0-99: smallest value -> 99, largest -> 0; ties averaged."""
    return s.rank(pct=True, ascending=False, method='average') * 99.0


sub = visits_summary.loc[iq_selection, IQ_COMPONENTS]
percentiles = sub.apply(_ranking_percentile)
visits_summary['iq_score'] = np.nan
visits_summary.loc[iq_selection, 'iq_score'] = percentiles.mean(axis=1)

print(f"n visits with iq_score: {visits_summary['iq_score'].notna().sum()}")


<a id='results'></a>
## Results & Plots

### Keith's original plots

Per-detector distributions, per-visit instrument-contribution histogram, nightly trending (instrument and measured), and the moment-score time-series with and without a donut-blur cut. All pages are written to a single PDF.

In [ ]:
# Keith's original plots -> a single multi-page PDF.
# Order: per-detector distributions, per-visit instrument contribution,
# nightly trending (instrument and measured), moment-score nightly
# time-series.
from matplotlib.backends.backend_pdf import PdfPages

keith_pdf_path = figures_dir / f'keith_plots_{day_obs_min}-{day_obs_max}.pdf'
keith_pdf = PdfPages(keith_pdf_path) if SAVE_FIGURES else None
try:
    if ccdvisits is None:
        print('Skipping per-detector plot (LOAD_CCDVISITS_LITE is False)')
    else:
        #selection = (ccdvisits['day_obs'] >= 20251130) & (ccdvisits['day_obs'] <= 20251202)
        selection = (ccdvisits['day_obs'] >= day_obs_min) & (ccdvisits['day_obs'] <= day_obs_max)
        #print(np.sum(selection) / 189.)
    
    
        bins = np.linspace(-0.4, 0.4, 81)
    
        plt.figure()
        plt.hist(ccdvisits['psf_e'][selection], bins=bins, density=True, histtype='step', lw=2)# label='PSF e')
        #plt.hist(ccdvisits['psf_e1'][selection], bins=bins, density=True, histtype='step', lw=2, label='PSF e1')
        #plt.hist(ccdvisits['psf_e2'][selection], bins=bins, density=True, histtype='step', lw=2, label='PSF e2')
    
        plt.axvline(np.nanmedian(ccdvisits['psf_e'][selection]), ls='--', lw=1, c='0.5', label='Median = %.2f'%(np.nanmedian(ccdvisits['psf_e'][selection])))
        print(np.nanmedian(ccdvisits['psf_e'][selection]))
    
        plt.legend()
        plt.xlabel('Ellipticity')
        plt.ylabel('PDF')
        #plt.xlim(bins[0], bins[-1])
        plt.xlim(0, bins[-1])
        plt.ylim(0., plt.ylim()[-1])
        plt.title(f"{min(ccdvisits['day_obs'][selection])} - {max(ccdvisits['day_obs'][selection])}")
        if keith_pdf is not None:
            keith_pdf.savefig(plt.gcf(), bbox_inches='tight')
        plt.show()
    
    if ccdvisits is None:
        print('Skipping per-detector plot (LOAD_CCDVISITS_LITE is False)')
    else:
        bins = np.linspace(0.5, 2., 51)
    
        plt.figure()
    
        for band in bands:
            selection = (ccdvisits["band"] == band)
            plt.hist(ccdvisits["psf_fwhm"][selection], bins=bins, color=colors[band], density=True, histtype="step", lw=2, label=band)
    
        plt.grid()
        plt.xlabel('PSF FWHM (arcsec)')
        plt.ylabel('Fraction of Sensors')
        plt.legend()
        plt.xlim(np.min(bins), np.max(bins))
        plt.title(f'{day_obs_min}-{day_obs_max}')
        plt.tight_layout()
        if keith_pdf is not None:
            keith_pdf.savefig(plt.gcf(), bbox_inches='tight')
        plt.show()
    
    if ccdvisits is None:
        print('Skipping per-detector plot (LOAD_CCDVISITS_LITE is False)')
    else:
        bins = np.linspace(0.5, 2.5, 51)
    
        plt.figure()
    
        for band in bands:
            selection = (ccdvisits["band"] == band)
            plt.hist(ccdvisits["psf_fwhm"][selection], bins=bins, color=colors[band], density=True, histtype="step", cumulative=True, lw=2, label=band)
        plt.grid()
        plt.xlabel('PSF FWHM (arcsec)')
        plt.ylabel('Fraction of Sensors')
        plt.legend(loc='lower right')
        plt.xlim(np.min(bins), np.max(bins))
        plt.ylim(0., 1.)
        plt.title(f'{day_obs_min}-{day_obs_max}')
        plt.tight_layout()
        if keith_pdf is not None:
            keith_pdf.savefig(plt.gcf(), bbox_inches='tight')
        plt.show()
    
    if ccdvisits is None:
        print('Skipping per-detector plot (LOAD_CCDVISITS_LITE is False)')
    else:
        log = True
        bins = np.linspace(0, 0.2, 1001)
    
        selection = np.isfinite(ccdvisits['moment_score'])
        selection = selection & (ccdvisits['band'] != 'y') # Exclude y filter
        selection = selection & ~((ccdvisits['day_obs'] == 20260424) & (ccdvisits['seq_num'] < 398)) # Remove corrupted period due to lack of hexapod compensation
    
        plt.figure()
        plt.hist(ccdvisits['moment_score'][selection], bins=bins, cumulative=-1, density=True, histtype='step', color='black')
    
        thresholds = 4. * np.array([0.002, 0.005, 0.02])
        for threshold in thresholds:
            plt.axvline(threshold, color='0.5', ls='--', lw=1)
    
            fraction = np.sum(
                (ccdvisits['moment_score'][selection] > threshold) / float(len(ccdvisits['moment_score'][selection]))
            )
            plt.axhline(fraction, color='0.5', ls='--', lw=1)
    
        plt.xlim(bins[0], 0.2)
    
        plt.axvspan(0., 4. * 0.002, color='tab:blue', alpha=0.1)
        plt.axvspan(4. * 0.002, 4. * 0.005, color='tab:green', alpha=0.1)
        plt.axvspan(4. * 0.005, 4. * 0.02, color='yellow', alpha=0.1)
        plt.axvspan(4. * 0.02, 4. * 0.05, color='tab:red', alpha=0.1)
    
        plt.text(0.008 - 0.001, 0.9, 'Excellent', rotation=90, fontsize=8, verticalalignment='top', horizontalalignment='right', color='0.5')
        plt.text(0.02 - 0.001, 0.9, 'Good', rotation=90, fontsize=8, verticalalignment='top', horizontalalignment='right', color='0.5')
        plt.text(0.08 - 0.001, 0.9, 'Marginal', rotation=90, fontsize=8, verticalalignment='top', horizontalalignment='right', color='0.5')
        plt.text(0.08 + 0.001, 0.9, 'Poor', rotation=90, fontsize=8, verticalalignment='top', horizontalalignment='left', color='0.5')
    
        if log:
            plt.ylim(0.0001, 1.)
            plt.yscale('log')
        else:
            plt.ylim(0., 1.)
        plt.xlabel('Moment Score')
        plt.ylabel('1 - CDF')
        plt.title(f'{day_obs_min} - {day_obs_max}')
        plt.tight_layout()
        if keith_pdf is not None:
            keith_pdf.savefig(plt.gcf(), bbox_inches='tight')
        plt.show()
    
    if ccdvisits is None:
        print('Skipping per-detector plot (LOAD_CCDVISITS_LITE is False)')
    else:
        log = True
        bins = np.linspace(0, 0.2, 1001)
    
        selection_donut_blur = (ccdvisits['donut_blur_fwhm'] < 0.8)
        selection = np.isfinite(ccdvisits['moment_score'])
        selection = selection & (ccdvisits['band'] != 'y') # Exclude y filter
        selection = selection & ~((ccdvisits['day_obs'] == 20260424) & (ccdvisits['seq_num'] < 398)) # Remove corrupted period due to lack of hexapod compensation
    
        plt.figure()
        plt.hist(ccdvisits['moment_score'][selection_donut_blur & selection], bins=bins, cumulative=-1, density=True, histtype='step', color='black', label='Donut Blur < 0.8')
        plt.hist(ccdvisits['moment_score'][~selection_donut_blur & selection], bins=bins, cumulative=-1, density=True, histtype='step', color='black', ls='--', label='Donut Blur > 0.8')
    
        thresholds = 4. * np.array([0.002, 0.005, 0.02])
        for threshold in thresholds:
            plt.axvline(threshold, color='0.5', ls='--', lw=1)
    
            #fraction = np.sum(ccdvisits['moment_score'] > threshold) / float(len(ccdvisits['moment_score']))
            #plt.axhline(fraction, color='0.5', ls='--', lw=1)
    
        plt.xlim(bins[0], 0.2)
    
        plt.axvspan(0., 4. * 0.002, color='tab:blue', alpha=0.1)
        plt.axvspan(4. * 0.002, 4. * 0.005, color='tab:green', alpha=0.1)
        plt.axvspan(4. * 0.005, 4. * 0.02, color='yellow', alpha=0.1)
        plt.axvspan(4. * 0.02, 4. * 0.05, color='tab:red', alpha=0.1)
    
        plt.text(0.008 - 0.001, 0.9, 'Excellent', rotation=90, fontsize=8, verticalalignment='top', horizontalalignment='right', color='0.5')
        plt.text(0.02 - 0.001, 0.9, 'Good', rotation=90, fontsize=8, verticalalignment='top', horizontalalignment='right', color='0.5')
        plt.text(0.08 - 0.001, 0.9, 'Marginal', rotation=90, fontsize=8, verticalalignment='top', horizontalalignment='right', color='0.5')
        plt.text(0.08 + 0.001, 0.9, 'Poor', rotation=90, fontsize=8, verticalalignment='top', horizontalalignment='left', color='0.5')
    
        if log:
            plt.ylim(0.0001, 1.)
            plt.yscale('log')
        else:
            plt.ylim(0., 1.)
        plt.legend()
        plt.xlabel('Moment Score')
        plt.ylabel('1 - CDF')
        plt.title(f'{day_obs_min} - {day_obs_max}')
        plt.tight_layout()
        if keith_pdf is not None:
            keith_pdf.savefig(plt.gcf(), bbox_inches='tight')
        plt.show()
    
    selection = (visits_summary['band'] != "y") & ~visits_summary['observation_reason'].str.contains('filter_change_close_loop')
    selection = selection & ~((visits_summary['day_obs'] == 20260424) & (visits_summary['seq_num'] < 398)) # Remove corrupted period due to lack of hexapod compensation
    
    bins = np.linspace(0., 1.2, 61)
    
    plt.figure()
    
    plt.hist(visits_summary['psf_fwhm_95_05'][selection], bins=bins, histtype='step', density=True, lw=2, color='tab:blue', label='Variation across FoV')
    plt.hist(visits_summary['sys_50'][selection], bins=bins, histtype='step', density=True, lw=2, color='tab:red', label='Delivered - Estimated Atmosphere')
    plt.hist(visits_summary['aos_cam_fwhm'][selection], bins=bins, histtype='step', density=True, lw=2, color='tab:purple', label='Optics + Camera Diffusion')
    plt.axvspan(0., 0.45, color='tab:green', alpha=0.1)
    plt.xlim(0., 1.2)
    
    print(np.nanmedian(visits_summary['psf_fwhm_95_05'][selection]))
    print(np.nanmedian(visits_summary['sys_50'][selection]))
    print(np.nanmedian(visits_summary['aos_cam_fwhm'][selection]))
    
    plt.axvline(np.nanmedian(visits_summary['psf_fwhm_95_05'][selection]), ls='--', lw=1, c='tab:blue')
    plt.axvline(np.nanmedian(visits_summary['sys_50'][selection]), ls='--', lw=1, c='tab:red')
    plt.axvline(np.nanmedian(visits_summary['aos_cam_fwhm'][selection]), ls='--', lw=1, c='tab:purple')
    
    plt.xlabel('Estimated Instrument Contribution (arcsec)')
    plt.ylabel('PDF')
    plt.legend(loc='upper right')
    plt.title(f'{day_obs_min} - {day_obs_max}')
    plt.tight_layout()
    if keith_pdf is not None:
        keith_pdf.savefig(plt.gcf(), bbox_inches='tight')
    plt.show()
    
    selection = np.tile(True, len(day_obs_summary))
    
    #xticks = np.arange(len(day_obs_summary[selection]))
    xticks = np.array([dayObsToTime(_).mjd for _ in day_obs_summary['day_obs'][selection]])
    
    fig, ax = plt.subplots(figsize=(15, 6), dpi=200)
    
    variation = ax.errorbar(
        xticks - 0.15, day_obs_summary['psf_fwhm_95_05_50'][selection],
        yerr=np.array([
            day_obs_summary['psf_fwhm_95_05_50'][selection] - day_obs_summary['psf_fwhm_95_05_low'][selection], 
            day_obs_summary['psf_fwhm_95_05_high'][selection] - day_obs_summary['psf_fwhm_95_05_50'][selection]
        ]), 
        fmt='_', c='tab:blue', label='Variation Across FoV'
        #label='sqrt(fwhm_95^2 - fwhm_5^2)'
    )
    aos = ax.errorbar(
        xticks + 0., day_obs_summary['aos_cam_fwhm_50'][selection],
        yerr=np.array([day_obs_summary['aos_cam_fwhm_50'][selection] - day_obs_summary['aos_cam_fwhm_low'][selection], 
                       day_obs_summary['aos_cam_fwhm_high'][selection] - day_obs_summary['aos_cam_fwhm_50'][selection]]), 
        fmt='_', c='tab:purple', label='Optics + Camera Diffusion'
        #label='sqrt(aos_fwhm^2 + cam_fwhm^2)'
    )
    sys = ax.errorbar(
        xticks + 0.15, day_obs_summary['sys_50_50'][selection],
        yerr=np.array([day_obs_summary['sys_50_50'][selection] - day_obs_summary['sys_50_low'][selection], 
                       day_obs_summary['sys_50_high'][selection] - day_obs_summary['sys_50_50'][selection]]), 
        fmt='_', c='tab:red', label='Delivered - Estimated Atmosphere'
        #label='sqrt(fwhm_50^2 - (donut_blur^2 - cam_fwhm^2))'
    )
    
    
    ellipticity = ax.errorbar(
        xticks + 0., day_obs_summary['psf_e_50_50'][selection],
        yerr=np.array([day_obs_summary['psf_e_50_50'][selection] - day_obs_summary['psf_e_50_low'][selection], 
                       day_obs_summary['psf_e_50_high'][selection] - day_obs_summary['psf_e_50_50'][selection]]), 
        fmt='_', c='tab:orange', label='PSF Ellipticity'
        #label='psf_e'
    )
    
    
    first_legend = ax.legend(handles=[variation, aos, sys], loc='upper left', title='Estimated Instrument Contribution')
    #first_legend = ax.legend(handles=[variation, aos], loc='upper left', title='Estimated System Contribution')
    #second_legend = ax.legend(handles=[psf, atm, ellipticity], loc='upper right', title='Measured Quantities')
    second_legend = ax.legend(handles=[ellipticity], loc='upper right', title='Measured Quantities')
    ax.add_artist(first_legend)
    
    ax.set_xticks(xticks, day_obs_summary['day_obs'][selection], rotation=90, fontsize=7.5)
    #ax.set_ylim(0, plt.ylim()[-1])
    #ax.set_ylim(0, 2.5)
    ax.set_xlim(xticks[0] - 1, xticks[-1] + 1)
    ax.set_ylim(0., 1.0)
    #ax.set_ylim(0.3, 0.5)
    #for y in [0.04, 0.2, 0.4, 0.6]:
    #    plt.axhline(y, c='0.75', ls='--', zorder=0)
    
    #ax.axhline(1., c='0.25', ls=':', zorder=0)
    ax.axhline(0.45, c='tab:green', ls=':', zorder=0)
    ax.axhline(0.04, c='tab:orange', ls=':', zorder=0)
    
    plt.axhspan(0.04, 0.45, color='tab:green', alpha=0.1)
    plt.axhspan(0., 0.04, color='tab:orange', alpha=0.1)
    
    ax.set_ylabel('Image Quality Metric (arcsec)')
    plt.title('10th, 50th, 90th Percentiles of Per-visit Quantities for Each Night')
    plt.grid()
    plt.minorticks_off()
    if keith_pdf is not None:
        keith_pdf.savefig(plt.gcf(), bbox_inches='tight')
    fig.tight_layout()
    plt.show()
    
    selection = np.tile(True, len(day_obs_summary))
    
    #xticks = np.arange(len(day_obs_summary[selection]))
    xticks = np.array([dayObsToTime(_).mjd for _ in day_obs_summary['day_obs'][selection]])
    
    fig, ax = plt.subplots(figsize=(15, 6), dpi=200)
    psf = ax.errorbar(
        xticks - 0.075, day_obs_summary['psf_fwhm_50_50'][selection],
        yerr=np.array([day_obs_summary['psf_fwhm_50_50'][selection] - day_obs_summary['psf_fwhm_50_low'][selection], 
                       day_obs_summary['psf_fwhm_50_high'][selection] - day_obs_summary['psf_fwhm_50_50'][selection]]), 
        fmt='_', c='black', label='Delivered Median PSF FWHM' 
        #label='fwhm_50'
    )
    atm = ax.errorbar(
        xticks + 0.075, day_obs_summary['donut_blur_atm_fwhm_50'][selection],
        yerr=np.array([day_obs_summary['donut_blur_atm_fwhm_50'][selection] - day_obs_summary['donut_blur_atm_fwhm_low'][selection], 
                       day_obs_summary['donut_blur_atm_fwhm_high'][selection] - day_obs_summary['donut_blur_atm_fwhm_50'][selection]]), 
        fmt='_', c='0.5', label='Estimated Atmosphere'
        #label='sqrt(donut_blur^2 - cam_fwhm^2)'
    )
    
    
    
    #first_legend = ax.legend(handles=[variation, aos, sys], loc='upper left', title='Estimated System Contribution')
    #first_legend = ax.legend(handles=[variation, aos], loc='upper left', title='Estimated System Contribution')
    #second_legend = ax.legend(handles=[psf, atm, ellipticity], loc='upper right', title='Measured Quantities')
    second_legend = ax.legend(handles=[psf, atm], loc='upper right', title='Measured Quantities')
    ax.add_artist(second_legend)
    
    ax.set_xticks(xticks, day_obs_summary['day_obs'][selection], rotation=90, fontsize=7.5)
    ax.set_xlim(xticks[0] - 1, xticks[-1] + 1)
    #ax.set_ylim(0, plt.ylim()[-1])
    #ax.set_ylim(0, 2.5)
    ax.set_ylim(0., 2.0)
    #ax.set_ylim(0.3, 0.5)
    #for y in [0.04, 0.2, 0.4, 0.6]:
    #    plt.axhline(y, c='0.75', ls='--', zorder=0)
    
    #ax.axhline(1., c='0.25', ls=':', zorder=0)
    
    ax.set_ylabel('Image Quality Metric (arcsec)')
    plt.title('10th, 50th, 90th Percentiles of Per-visit Quantities for Each Night')
    plt.grid()
    plt.minorticks_off()
    fig.tight_layout()
    if keith_pdf is not None:
        keith_pdf.savefig(plt.gcf(), bbox_inches='tight')
    plt.show()
    
    if ccdvisits is None:
        print('Skipping per-detector plot (LOAD_CCDVISITS_LITE is False)')
    else:
        APPLY_DONUT_BLUR_SELECTION = False
        DONUT_BLUR_THRESHOLD = 0.8 # arcsec
    
        if APPLY_DONUT_BLUR_SELECTION:
            selection_donut_blur = (ccdvisits['donut_blur_fwhm'] < DONUT_BLUR_THRESHOLD)
        else:
            selection_donut_blur = np.tile(True, len(ccdvisits))
    
        selection_donut_blur = selection_donut_blur & (ccdvisits['band'] != 'y') # Exclude y filter
        selection_donut_blur = selection_donut_blur & ~((ccdvisits['day_obs'] == 20260424) & (ccdvisits['seq_num'] < 398)) # Remove corrupted period due to lack of hexapod compensation
    
        groups = ccdvisits[selection_donut_blur].groupby('day_obs')
        day_obs_summary_moments = pd.DataFrame({
            'day_obs': groups['day_obs'].first(),
            'n_ccdvisits': groups['day_obs'].count(),
            'moment_score_low': groups['moment_score'].agg(lambda x: (x <= 4. * 0.002).mean()),
            'moment_score_002': groups['moment_score'].agg(lambda x: (x > 4. * 0.002).mean()),
            'moment_score_005': groups['moment_score'].agg(lambda x: (x > 4. * 0.005).mean()),
            'moment_score_02': groups['moment_score'].agg(lambda x: (x > 4. * 0.02).mean()),
        })
    
    
        selection = np.tile(True, len(day_obs_summary_moments))
        xticks = np.array([dayObsToTime(_).mjd for _ in day_obs_summary_moments['day_obs'][selection]])
    
        fig, ax = plt.subplots(figsize=(15, 6), dpi=200)
    
        ax.scatter(xticks + 0., day_obs_summary_moments['moment_score_low'][selection], c='tab:blue', label='Excellent (Moment Score < 0.008)')
        ax.scatter(xticks + 0., 1. - day_obs_summary_moments['moment_score_005'][selection], c='tab:green', label='Excellent + Good (Moment Score < 0.02)')
    
        ax.legend(loc='lower right')
    
        ax.set_xticks(xticks, day_obs_summary_moments['day_obs'][selection], rotation=90, fontsize=7.5)
        ax.set_xlim(xticks[0] - 1, xticks[-1] + 1)
        ax.set_ylim(0., 1.0)
        ax.set_ylabel('Fraction of Detector Images\nMeeting Criteria')
        plt.grid()
        plt.minorticks_off()
        if APPLY_DONUT_BLUR_SELECTION:
            plt.title(f'Donut Blur < {DONUT_BLUR_THRESHOLD}')
        plt.tight_layout()
        if keith_pdf is not None:
            keith_pdf.savefig(plt.gcf(), bbox_inches='tight')
        plt.show()
    
    if ccdvisits is None:
        print('Skipping per-detector plot (LOAD_CCDVISITS_LITE is False)')
    else:
        APPLY_DONUT_BLUR_SELECTION = True
        DONUT_BLUR_THRESHOLD = 0.8 # arcsec
    
        if APPLY_DONUT_BLUR_SELECTION:
            selection_donut_blur = (ccdvisits['donut_blur_fwhm'] < DONUT_BLUR_THRESHOLD)
        else:
            selection_donut_blur = np.tile(True, len(ccdvisits))
    
        selection_donut_blur = selection_donut_blur & (ccdvisits['band'] != 'y') # Exclude y filter
        selection_donut_blur = selection_donut_blur & ~((ccdvisits['day_obs'] == 20260424) & (ccdvisits['seq_num'] < 398)) # Remove corrupted period due to lack of hexapod compensation
    
        groups = ccdvisits[selection_donut_blur].groupby('day_obs')
        day_obs_summary_moments = pd.DataFrame({
            'day_obs': groups['day_obs'].first(),
            'n_ccdvisits': groups['day_obs'].count(),
            'moment_score_low': groups['moment_score'].agg(lambda x: (x <= 4. * 0.002).mean()),
            'moment_score_002': groups['moment_score'].agg(lambda x: (x > 4. * 0.002).mean()),
            'moment_score_005': groups['moment_score'].agg(lambda x: (x > 4. * 0.005).mean()),
            'moment_score_02': groups['moment_score'].agg(lambda x: (x > 4. * 0.02).mean()),
        })
    
    
        selection = np.tile(True, len(day_obs_summary_moments))
        xticks = np.array([dayObsToTime(_).mjd for _ in day_obs_summary_moments['day_obs'][selection]])
    
        fig, ax = plt.subplots(figsize=(15, 6), dpi=200)
    
        ax.scatter(xticks + 0., day_obs_summary_moments['moment_score_low'][selection], c='tab:blue', label='LOW (Moment Score < 0.008)')
        ax.scatter(xticks + 0., 1. - day_obs_summary_moments['moment_score_005'][selection], c='tab:green', label='LOW + MEDIUM (Moment Score < 0.02)')
    
        ax.legend(loc='lower right')
    
        ax.set_xticks(xticks, day_obs_summary_moments['day_obs'][selection], rotation=90, fontsize=7.5)
        ax.set_xlim(xticks[0] - 1, xticks[-1] + 1)
        ax.set_ylim(0., 1.0)
        ax.set_ylabel('Fraction of Detector Images\nMeeting Criteria')
        plt.grid()
        plt.minorticks_off()
        if APPLY_DONUT_BLUR_SELECTION:
            plt.title(f'Donut Blur < {DONUT_BLUR_THRESHOLD}')
        plt.tight_layout()
        if keith_pdf is not None:
            keith_pdf.savefig(plt.gcf(), bbox_inches='tight')
        plt.show()
finally:
    if keith_pdf is not None:
        keith_pdf.close()
        print(f'Wrote {keith_pdf_path}')


### IQ score pipeline

Validation (per-visit corner plot) and normalized inverse CDFs for each IQ component first, then per-component nightly time-series (errorbar + violin), then the combined IQ score histograms and trends. All pages are written to a single PDF.

In [ ]:
# IQ score pipeline plots -> a single multi-page PDF.
# Order:
#   1. Validation: per-visit corner plot (2 pages: all visits / donut_blur<0.8)
#   2. Normalized inverse cumulative distributions for each IQ component
#   3. Per-component nightly time series: 10/50/90 errorbar + violin
#   4. Combined IQ score: per-band histogram, errorbar, violin
from matplotlib.backends.backend_pdf import PdfPages

iq_pdf_path = figures_dir / f'iq_plots_{day_obs_min}-{day_obs_max}.pdf'
iq_pdf = PdfPages(iq_pdf_path) if SAVE_FIGURES else None
try:
    # --- Validation: corner plot ---
    corner_columns = [
        'psf_fwhm_50',
        'donut_blur_fwhm',
        'aos_cam_fwhm',
        'psf_e_50',
        'psf_fwhm_95_05',
        'psf_e_95_05',
        'log10_moment_score',
    ]
    corner_labels = [
        'PSF FWHM',
        'Donut Blur',
        'AOS+Cam FWHM',
        'PSF Ellipticity',
        '95-5% FWHM',
        '95-5% Ellipticity',
        'log10(Moment Score)',
    ]
    
    base_selection = (visits_summary['band'] != 'y') \
        & ~visits_summary['observation_reason'].str.contains('filter_change_close_loop')
    base_selection = base_selection & ~(
        (visits_summary['day_obs'] == 20260424) & (visits_summary['seq_num'] < 398)
    )
    
    # Two pages: full sample, and donut_blur_fwhm < 0.8 (matches the IQ summary plots).
    DONUT_BLUR_THRESHOLD = 0.8
    panels = [
        ('all visits', base_selection),
        (f'donut_blur_fwhm < {DONUT_BLUR_THRESHOLD}',
         base_selection & (visits_summary['donut_blur_fwhm'] < DONUT_BLUR_THRESHOLD)),
    ]
    
    for tag, sel in panels:
        sub = visits_summary[sel].copy()
        # log10 to flatten the moment_score tail; clip to avoid -inf at zero.
        sub['log10_moment_score'] = np.log10(
            sub['moment_score'].clip(lower=1e-8)
        )
        fig, axes = corner_plot(sub, corner_columns, labels=corner_labels)
        fig.suptitle(
            f'Per-visit IQ correlations  {day_obs_min}-{day_obs_max}  '
            f'[{tag}, n={len(sub)}]',
            y=1.0,
        )
        fig.tight_layout()
        if iq_pdf is not None:
            iq_pdf.savefig(fig, bbox_inches='tight')
        plt.show()
    
    # --- Normalized inverse CDFs of IQ components ---
    # Normalized inverse cumulative distributions for each IQ score input.
    #
    # The percentile rank used in the combined IQ score is just
    # `99 * (1 - F(x))` evaluated at the visit's component value. These
    # panels visualize that mapping per component so it's obvious where the
    # spread lies and how each component contributes to the score.
    n = len(IQ_COMPONENTS)
    ncols = min(3, n)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(
        nrows, ncols, figsize=(4 * ncols, 3 * nrows), squeeze=False,
    )
    
    for k, col in enumerate(IQ_COMPONENTS):
        ax = axes.flat[k]
        s = visits_summary.loc[iq_selection, col].dropna()
        if len(s) == 0:
            ax.set_visible(False)
            continue
        sorted_vals = np.sort(s.to_numpy())
        n_vals = len(sorted_vals)
        survival = np.arange(n_vals, 0, -1) / n_vals  # 1 - F(x)
        ax.plot(sorted_vals, survival, lw=1.5, color='tab:blue')
        ax.set_xlabel(col)
        ax.set_ylabel('1 - F(x)')
        ax.set_ylim(0, 1)
        ax.grid(True)
        # Mirror axis: percentile rank = 99 * (1 - F(x))
        ax2 = ax.twinx()
        ax2.set_ylim(0, 99)
        ax2.set_ylabel('Percentile rank (99 = best)', fontsize=8)
    
    # Hide any unused panels.
    for k in range(len(IQ_COMPONENTS), nrows * ncols):
        axes.flat[k].set_visible(False)
    
    fig.suptitle(
        f'Normalized inverse CDF of IQ components  '
        f'{day_obs_min}-{day_obs_max}'
    )
    fig.tight_layout()
    if iq_pdf is not None:
        iq_pdf.savefig(fig, bbox_inches='tight')
    plt.show()
    
    # --- Per-component 10/50/90 errorbar time-series ---
    # 10/50/90 percentile per-night time-series for each IQ component
    # individually, computed over the same `iq_selection` ensemble used
    # for the combined IQ score.
    iq_component_plots = [
        ('aos_cam_fwhm',  'AOS + Camera FWHM (arcsec)',  'tab:purple'),
        ('psf_fwhm_95_05', '95-5% PSF FWHM (arcsec)',    'tab:blue'),
        ('psf_e_50',      'PSF Ellipticity (median)',    'tab:orange'),
        ('psf_e_95_05',   '95-5% PSF Ellipticity',       'tab:red'),
    ]
    if USE_SHAPELETS_SCORE and 'shapelets_score' in visits_summary.columns \
            and visits_summary['shapelets_score'].notna().any():
        iq_component_plots.append(
            ('shapelets_score', 'Shapelets Score (TQ)', 'tab:green')
        )
    
    for col, label, color in iq_component_plots:
        groups = visits_summary[iq_selection].groupby('day_obs')
        trend = pd.DataFrame({
            'day_obs': groups['day_obs'].first(),
            'low':    groups[col].quantile(0.10),
            'median': groups[col].quantile(0.50),
            'high':   groups[col].quantile(0.90),
        })
        xticks = np.array([dayObsToTime(d).mjd for d in trend['day_obs']])
    
        fig, ax = plt.subplots(figsize=(15, 6), dpi=200)
        ax.errorbar(
            xticks, trend['median'],
            yerr=np.array([
                trend['median'] - trend['low'],
                trend['high']   - trend['median'],
            ]),
            fmt='_', c=color, label=label,
        )
        ax.set_xticks(xticks, trend['day_obs'], rotation=90, fontsize=7.5)
        ax.set_xlim(xticks[0] - 1, xticks[-1] + 1)
        ax.set_ylim(0, ax.get_ylim()[1])
        ax.set_ylabel(label)
        ax.legend(loc='upper right')
        plt.title(f'10th, 50th, 90th Percentiles per Night: {label}')
        plt.grid()
        plt.minorticks_off()
        fig.tight_layout()
        if iq_pdf is not None:
            iq_pdf.savefig(plt.gcf(), bbox_inches='tight')
        plt.show()
    
    # --- Per-component violin time-series ---
    # Violin counterpart of the per-component nightly time-series above.
    for col, label, color in iq_component_plots:
        fig = plot_violin_timeseries(
            visits_summary, col, label, sel=iq_selection, color=color,
        )
        if fig is not None and iq_pdf is not None:
            iq_pdf.savefig(fig, bbox_inches='tight')
        plt.show()
    
    # --- Combined IQ score histogram per band ---
    bins = np.linspace(0, 100, 51)
    
    plt.figure()
    for band in bands:
        sel = iq_selection & (visits_summary['band'] == band)
        if not sel.any():
            continue
        plt.hist(
            visits_summary['iq_score'][sel], bins=bins, color=colors[band],
            density=True, histtype='step', lw=2, label=band,
        )
    plt.grid()
    plt.xlabel('Combined IQ score  (99 = best, 0 = worst)')
    plt.ylabel('PDF')
    plt.legend()
    plt.xlim(0, 100)
    plt.title(f'{day_obs_min}-{day_obs_max}')
    plt.tight_layout()
    if iq_pdf is not None:
        iq_pdf.savefig(plt.gcf(), bbox_inches='tight')
    plt.show()
    
    # --- Combined IQ score 10/50/90 errorbar time-series ---
    iq_groups = visits_summary[iq_selection].groupby('day_obs')
    iq_trend = pd.DataFrame({
        'day_obs': iq_groups['day_obs'].first(),
        'n_visits': iq_groups['iq_score'].count(),
        'iq_score_low':  iq_groups['iq_score'].quantile(0.10),
        'iq_score_50':   iq_groups['iq_score'].quantile(0.50),
        'iq_score_high': iq_groups['iq_score'].quantile(0.90),
    })
    
    xticks = np.array([dayObsToTime(d).mjd for d in iq_trend['day_obs']])
    
    fig, ax = plt.subplots(figsize=(15, 6), dpi=200)
    ax.errorbar(
        xticks, iq_trend['iq_score_50'],
        yerr=np.array([
            iq_trend['iq_score_50'] - iq_trend['iq_score_low'],
            iq_trend['iq_score_high'] - iq_trend['iq_score_50'],
        ]),
        fmt='_', c='tab:purple', label='Combined IQ score',
    )
    ax.set_xticks(xticks, iq_trend['day_obs'], rotation=90, fontsize=7.5)
    ax.set_xlim(xticks[0] - 1, xticks[-1] + 1)
    ax.set_ylim(0, 100)
    ax.set_ylabel('Combined IQ score  (99 = best)')
    ax.legend(loc='lower right')
    plt.title('10th, 50th, 90th Percentiles of Combined IQ Score per Night')
    plt.grid()
    plt.minorticks_off()
    fig.tight_layout()
    if iq_pdf is not None:
        iq_pdf.savefig(plt.gcf(), bbox_inches='tight')
    plt.show()
    
    # --- Combined IQ score violin time-series ---
    fig = plot_violin_timeseries(
        visits_summary, 'iq_score', 'Combined IQ score  (99 = best)',
        sel=iq_selection, color='tab:purple',
    )
    if fig is not None and iq_pdf is not None:
        iq_pdf.savefig(fig, bbox_inches='tight')
    plt.show()
finally:
    if iq_pdf is not None:
        iq_pdf.close()
        print(f'Wrote {iq_pdf_path}')
